# Day 6 - Error Analysis

Systematically buckets every false negative (FN) and false positive (FP)
from all three evaluated models by entity type, draws a stratified sample for manual
annotation, and produces the error-distribution figure used in the report.

**Model roles**
| Approach | Model | Role | Prediction file |
|---|---|---|---|
| A | DistilBERT-s42 | Lightweight encoder baseline | `predictions/encoder/distilbert_s42_predictions.jsonl` |
| B | DeBERTa-s7 | Best in-domain encoder | `predictions/encoder/deberta_s7_predictions.jsonl` |
| C | LLaMA | Prompted local LLM baseline | `predictions/llm/raw_outputs.jsonl` |

**Prerequisite outputs (read-only)**
| File | Produced by |
|---|---|
| `predictions/encoder/distilbert_s42_predictions.jsonl` | Day 5 - `05_evaluate_all.py` |
| `predictions/encoder/deberta_s7_predictions.jsonl` | Day 5 - `05_evaluate_all.py` |
| `predictions/llm/raw_outputs.jsonl` | Day 4 - `04_llm_inference.py` |
| `results/day5/results.json` | Day 5 - `05_evaluate_all.py` |

**Outputs written here**
| Artefact | Location |
|---|---|
| Error bucket counts | `results/day6/error_buckets.json` |
| Manual annotation sheets | `errors/manual_*.tsv` |
| Error distribution figure | `reports/figures/fig1_error_analysis.png` |


## 1  Setup

In [14]:
import json
import pathlib
import sys

# Resolve project root so the src package is importable from the notebook.
def find_project_root(start=None):
    current = pathlib.Path(start or pathlib.Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'pii_masking').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root containing src/pii_masking')


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.pii_masking.day6_error_analysis import (
    BucketResult,
    build_tsv_rows,
    collect_error_instances,
    compute_buckets,
    load_encoder_predictions,
    load_llm_predictions,
    markdown_table_str,
    plot_error_distribution,
    sample_distribution_str,
    save_tsv,
    stratified_sample,
    summary_table_str,
)

print("Imports OK  |  ROOT:", ROOT)


Imports OK  |  ROOT: E:\Projects\pii_masking


In [15]:
# Paths
APPROACH_A_FILE    = ROOT / "predictions" / "encoder" / "distilbert_s42_predictions.jsonl"
APPROACH_B_FILE    = ROOT / "predictions" / "encoder" / "deberta_s7_predictions.jsonl"
APPROACH_C_FILE    = ROOT / "predictions" / "llm"     / "raw_outputs.jsonl"
RESULTS_JSON       = ROOT / "results" / "day5" / "results.json"
RESULTS_DAY6_DIR   = ROOT / "results"  / "day6"
ERROR_BUCKETS_JSON = RESULTS_DAY6_DIR / "error_buckets.json"
ERRORS_DIR         = ROOT / "errors"
FIGURES_DIR        = ROOT / "reports" / "figures"
FIGURE_PNG         = FIGURES_DIR / "fig1_error_analysis.png"

# Create output directories up-front
for d in (RESULTS_DAY6_DIR, ERRORS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Output directories ready.")


Output directories ready.


## 2  Load predictions

In [16]:
records_a = load_encoder_predictions(APPROACH_A_FILE)
records_b = load_encoder_predictions(APPROACH_B_FILE)
records_c = load_llm_predictions(APPROACH_C_FILE)

print(f"Approach A (DistilBERT-s42) : {len(records_a):,} sentences")
print(f"Approach B (DeBERTa-s7)    : {len(records_b):,} sentences")
print(f"Approach C (LLaMA)         : {len(records_c):,} sentences")

# Sanity-check: first record from each approach
for label, records in {
    "Approach A (DistilBERT-s42)": records_a,
    "Approach B (DeBERTa-s7)": records_b,
    "Approach C (LLaMA)": records_c,
}.items():
    print(f"\n--- {label} sample ---")
    r = records[0]
    print("tokens   :", r["tokens"][:8], "...")
    print("gold_tags:", r["gold_tags"][:8], "...")
    print("pred_tags:", r["pred_tags"][:8], "...")


Approach A (DistilBERT-s42) : 3,650 sentences
Approach B (DeBERTa-s7)    : 3,650 sentences
Approach C (LLaMA)         : 3,650 sentences

--- Approach A (DistilBERT-s42) sample ---
tokens   : ['This', 'Is', 'What', 'the', 'Truth', 'Feels', 'Like', '"'] ...
gold_tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'] ...
pred_tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'] ...

--- Approach B (DeBERTa-s7) sample ---
tokens   : ['This', 'Is', 'What', 'the', 'Truth', 'Feels', 'Like', '"'] ...
gold_tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'] ...
pred_tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'] ...

--- Approach C (LLaMA) sample ---
tokens   : ['This', 'Is', 'What', 'the', 'Truth', 'Feels', 'Like', '"'] ...
gold_tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'] ...
pred_tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'] ...


## 3  Error bucket counts

Buckets are defined by the cross-product of direction (FN = missed entity, FP = spurious entity)
and entity type (PER, EMAIL).  Each span is compared at **exact** boundary level - both endpoints
and the label must match the gold annotation.

In [17]:
MODELS = {
    "DistilBERT-s42": compute_buckets(records_a),
    "DeBERTa-s7": compute_buckets(records_b),
    "LLaMA": compute_buckets(records_c),
}

print(summary_table_str(MODELS))


Bucket        DistilBERT-s        %    DeBERTa-s7        %         LLaMA        %
---------------------------------------------------------------------------------
FN_PER                  94    48.2%            84    42.0%         1,374     9.4%
FN_EMAIL                 4     2.0%             0     0.0%           707     4.8%
FP_PER                  95    48.7%           116    58.0%         9,261    63.3%
FP_EMAIL                 2     1.0%             0     0.0%         3,289    22.5%
---------------------------------------------------------------------------------
TOTAL                  195                    200                 14,631         


In [18]:
# Save to disk
payload = {
    "approach_A_distilbert_s42": MODELS["DistilBERT-s42"].to_dict(),
    "approach_B_deberta_s7": MODELS["DeBERTa-s7"].to_dict(),
    "approach_C_llama": MODELS["LLaMA"].to_dict(),
}
ERROR_BUCKETS_JSON.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Saved -> {ERROR_BUCKETS_JSON}")


Saved -> E:\Projects\pii_masking\results\day6\error_buckets.json


## 4  Error distribution figure

Grouped bar chart comparing Approach A (DistilBERT-s42), Approach B (DeBERTa-s7),
and Approach C (LLaMA). Error counts are recomputed each run from the prediction files above.


In [19]:
import matplotlib
matplotlib.use("Agg")          # headless; swap to "inline" if running interactively
import matplotlib.pyplot as plt

fig = plot_error_distribution(MODELS, FIGURE_PNG, dpi=150)

# Re-render inline in the notebook (Agg backend writes the file; re-open to display)
fig2, ax2 = plt.subplots(figsize=(8, 5))
img = plt.imread(str(FIGURE_PNG))
ax2.imshow(img)
ax2.axis("off")
plt.tight_layout()
plt.show()
print(f"Figure saved -> {FIGURE_PNG}")


Figure saved -> E:\Projects\pii_masking\reports\figures\fig1_error_analysis.png


C:\TEMP\ipykernel_49208\2554604117.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
print(markdown_table_str(MODELS))


| Bucket | DistilBERT-s42 | % Dis | DeBERTa-s7 | % DeB | LLaMA | % LLa |
|--------|--- | --- | --- | --- | --- | ---|
| FN_PER   |         94 |  48.2% |         84 |  42.0% |      1,374 |   9.4% |
| FN_EMAIL |          4 |   2.0% |          0 |   0.0% |        707 |   4.8% |
| FP_PER   |         95 |  48.7% |        116 |  58.0% |      9,261 |  63.3% |
| FP_EMAIL |          2 |   1.0% |          0 |   0.0% |      3,289 |  22.5% |
| **TOTAL** |        195 |  |        200 |  |     14,631 | |
